In [1]:
import pandas as pd

# Carregar o conjunto de dados em um DataFrame pandas
df = pd.read_csv("dados_manutencao.csv")

# Mostrar as primeiras 5 linhas
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Mostrar os nomes das colunas e seus tipos de dados
print(df.info())

| Data de Produção Acumulada   | Cod. Ordem   | Cod Recurso   | Cod Produto   | Qt. Total Acumulada Produzida até a data específica   | Qt. Acumulada Refugada até a data específica   | Qtd. Acumulada total Retrabalhada até a data específica   | Fator Un.   | Cód. Un.   | Descrição da massa (Composto)   | Consumo total de Massa Acumulada   | Tempo Restante para Manutenção   |
|:-----------------------------|:-------------|:--------------|:--------------|:------------------------------------------------------|:-----------------------------------------------|:----------------------------------------------------------|:------------|:-----------|:--------------------------------|:-----------------------------------|:---------------------------------|
| 2023-09-08                   | 4458603      | IJ-117        | SA02961       | 87                                                    | 98                                             | 14                                                        |

In [2]:
# Renomear colunas
df = df.rename(
    columns={
        "Qt. Total Acumulada Produzida até a data específica": "Qtd_Produzida",
        "Qt. Acumulada Refugada até a data específica": "Qtd_Refugada",
        "Qtd. Acumulada total Retrabalhada até a data específica": "Qtd_Retrabalhada",
    }
)

# Remover colunas
df = df.drop(
    [
        "Data de Produção Acumulada",
        "Cod. Ordem",
        "Cod Recurso",
        "Fator Un.",
        "Cód. Un.",
        "Descrição da massa (Composto)",
    ],
    axis=1,
)

# Converter a coluna `Cod Produto` em numéricas usando one-hot encoding
df = pd.get_dummies(df, columns=["Cod Produto"])

# Mostrar as primeiras 5 linhas
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Mostrar os nomes das colunas e seus tipos de dados
print(df.info())

| Qtd_Produzida   | Qtd_Refugada   | Qtd_Retrabalhada   | Consumo total de Massa Acumulada   | Tempo Restante para Manutenção   | Cod Produto_SA02004   | Cod Produto_SA02961   | Cod Produto_SA05780   |
|:----------------|:---------------|:-------------------|:-----------------------------------|:---------------------------------|:----------------------|:----------------------|:----------------------|
| 87              | 98             | 14                 | 0.909355                           | -157                             | False                 | True                  | False                 |
| 819             | 1              | 8                  | 0.796544                           | -195                             | False                 | False                 | True                  |
| 84              | 74             | 4                  | 0.686085                           | -153                             | False                 | False                 | True          

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# from skopt import BayesSearchCV
from sklearn.model_selection import RandomizedSearchCV

# Divida os dados em conjuntos de treinamento e teste
X = df.drop("Tempo Restante para Manutenção", axis=1)
y = df["Tempo Restante para Manutenção"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Crie o pipeline
numeric_features = [
    "Qtd_Produzida",
    "Qtd_Refugada",
    "Qtd_Retrabalhada",
    "Consumo total de Massa Acumulada",
]
categorical_features = [
    "Cod Produto_SA02004",
    "Cod Produto_SA02961",
    "Cod Produto_SA05780",
]

numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
categorical_transformer = Pipeline(
    steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

pipeline = Pipeline(
    steps=[("preprocessor", preprocessor), ("regressor", xgb.XGBRegressor())]
)

In [4]:
# Defina o espaço de busca de hiperparâmetros
param_space = {
    "regressor__n_estimators": (50, 200),  # Inteiros entre 50 e 200
    "regressor__max_depth": (3, 7),  # Inteiros entre 3 e 7
    "regressor__learning_rate": (
        1e-3,
        1e-1,
        "log-uniform",
    ),  # De 0.001 a 0.1 em escala logarítmica
    "regressor__subsample": (0.5, 1.0),  # De 0.5 a 1.0
    "regressor__colsample_bytree": (0.5, 1.0),  # De 0.5 a 1.0
}

# Crie e treine um modelo XGBoostRegressor usando BayesSearchCV com os hiperparâmetros fornecidos
# opt = BayesSearchCV(
#     pipeline,
#     param_space,
#     n_iter=50,  # Número de iterações da otimização Bayesiana
#     cv=5,  # Número de folds na validação cruzada
#     scoring='neg_mean_squared_error'
# )

opt = RandomizedSearchCV(
    pipeline,
    param_space,
    n_iter=50,  # Número de iterações da otimização Bayesiana
    cv=5,  # Número de folds na validação cruzada
    scoring="neg_mean_squared_error",
)

opt.fit(X_train, y_train)

/Users/josesilva/opt/anaconda3/envs/oxaala/lib/python3.9/site-packages/sklearn/model_selection/_search.py:320: UserWarning: The total space of parameters 48 is smaller than n_iter=50. Running 48 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/josesilva/opt/anaconda3/envs/oxaala/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
80 fits failed out of a total of 240.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
80 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/josesilva/opt/anaconda3/envs/oxaala/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, 

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('scaler',
                                                                                                StandardScaler())]),
                                                                               ['Qtd_Produzida',
                                                                                'Qtd_Refugada',
                                                                                'Qtd_Retrabalhada',
                                                                                'Consumo '
                                                                                'total '
                                                                                'de '
                                                                                'Massa '
                                                                                'Acumulada']),
                                                                              ('cat',
                                                                               Pipeline(steps=[('onehot',
                                                                                                OneHotEncoder(handle_unknown='ignore'))]),
                                                                               ['Cod '
                                                                                'Produto_SA02004',
                                                                                'Cod '
                                                                                'Pr...
                                                           multi_strategy=None,
                                                           n_estimators=None,
                                                           n_jobs=None,
                                                           num_parallel_tree=None,
                                                           random_state=None, ...))]),
                   n_iter=50,
                   param_distributions={'regressor__colsample_bytree': (0.5,
                                                                        1.0),
                                        'regressor__learning_rate': (0.001, 0.1,
                                                                     'log-uniform'),
                                        'regressor__max_depth': (3, 7),
                                        'regressor__n_estimators': (50, 200),
                                        'regressor__subsample': (0.5, 1.0)},
                   scoring='neg_mean_squared_error')

In [7]:
# Faça previsões no conjunto de teste e avalie o modelo usando as métricas MSE e R-quadrado
y_pred = opt.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Discretize a variável `Tempo Restante para Manutenção`
bins = [-float("inf"), -200, -100, float("inf")]
labels = ["Curto", "Médio", "Longo"]

y_test_discretized = pd.cut(y_test, bins=bins, labels=labels)
y_pred_discretized = pd.cut(y_pred, bins=bins, labels=labels)

In [8]:
# Calcule e exiba a matriz de confusão
cm = confusion_matrix(y_test_discretized, y_pred_discretized)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues")
plt.title("Matriz de Confusão")
plt.show()

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared: {r2:.2f}")

NameError: name 'confusion_matrix' is not defined